In [ ]:
import dask
# dask.config.set(scheduler="synchronous")

from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## Download Dataset

In [ ]:
from xradio.measurement_set import open_processing_set
from toolviper.utils.data import download, update
update()

download(file="twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")
ps_xdt = open_processing_set("twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")

download(file="twhya_selfcal_5chans_lsrk_xx_compare_weights.ps.zarr")
ps_single_pol_xdt = open_processing_set("twhya_selfcal_5chans_lsrk_xx_compare_weights.ps.zarr")


download(file="3c286_Band6_5chans_lsrk_compare_weights.ps.zarr")
ps_full_pol_xdt = open_processing_set("3c286_Band6_5chans_lsrk_compare_weights.ps.zarr")


ps_xdt.xr_ps.summary()

In [1]:

%load_ext autoreload
%autoreload 2

# import dask
# dask.config.set(scheduler="synchronous")

log_level = "DEBUG" 
#log_level = "INFO" 
log_to_file = False
log_params = { "logger_name": "main",
            "log_to_term": True,
            "log_level": log_level,
            "log_to_file": log_to_file,
            "log_file": "client.log",
            }

worker_logs = { "logger_name": "worker",
            "log_to_term": True,
            "log_level": log_level,
            "log_to_file": log_to_file,
            "log_file": "client_worker.log",
            }


from toolviper.dask.client import local_client

viper_client = local_client(cores=1, memory_limit="18GB", log_params=log_params, worker_log_params=worker_logs)
viper_client

import os
import numpy as np
from xradio.measurement_set import open_processing_set
from astroviper.distributed.imaging.image_cube_single_field import image_cube_single_field
from xradio.image import make_empty_sky_image

os.system("rm -rf twhya_selfcal_5chans_lsrk_compare_weights.img.zarr")
ps_single_pol_xdt = open_processing_set("twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")
combined_field_and_source_xds = ps_single_pol_xdt.xr_ps.get_combined_field_and_source_xds()
center_field_name = combined_field_and_source_xds.attrs["center_field_name"]
phase_direction = (
    combined_field_and_source_xds.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=center_field_name
    )
)

#print(ps_single_pol_xdt.xr_ps.get_freq_axis())

empty_img_xds = make_empty_sky_image(
        phase_center=phase_direction.values,
        image_size=[250,250],
        cell_size=np.array([-0.1,0.1]) * np.pi/(180 * 3600),
        frequency_coords=ps_single_pol_xdt.xr_ps.get_freq_axis().values,
        pol_coords=["I"],
        time_coords=[0],
    )

# image_cube_single_field

imaging_metadata_dict = image_cube_single_field(
    ps_store = "twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr",
    image_store = "twhya_selfcal_5chans_lsrk_compare_weights.img.zarr",
    image_params={
        "image_size": [250, 250],
        "cell_size": np.array([0.1, 0.1]) * np.pi/(180 * 3600),
        "phase_direction": phase_direction.values,
        "frequency_coords": ps_single_pol_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": ["I","Q"],
        "time_coords": [0],
    },
    imaging_weights_params={
        "weighting": "briggs",
        "robust": 0.5,
    },
    # imaging_weights_params={
    #     "weighting": "natural",
    # },
    iteration_control_params={
        "niter": 0,
        "nmajor": 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": 10,
    },
    gridder="prolate_spheroidal",
    deconvolver="hogbom",
    fft_padding="1.0",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    #image_data_variables_keep=["sky", "point_spread_function", "primary_beam"],
    image_data_variables_keep=["sky_model", "sky_residual", "sky_deconvolved", "point_spread_function", "primary_beam"],
    #image_data_variables_keep=[ "sky", "point_spread_function", "primary_beam"],
    processing_set_data_group_name="corrected",
    double_precision=True,
    thread_info=None,
    n_chunks=None,
    overwrite=True,
)

import xarray as xr
img_xds = xr.open_zarr("twhya_selfcal_5chans_lsrk_compare_weights.img.zarr")
img_xds

[2026-03-12 17:14:10,740]  WARNING        main:  It is recommended that the local cache directory be set using the dask_local_dir parameter. 
[2026-03-12 17:14:11,268]    DEBUG        main:  Loading plugin module: <class 'worker.DaskWorker'>
[2026-03-12 17:14:11,762]    DEBUG    worker_0:  Logger created on worker Worker-30990990-6bdc-4245-9df5-b43ea5ff7533,*,tcp://127.0.0.1:56005
[2026-03-12 17:14:11,763]     INFO        main:  Client <MenrvaClient: 'tcp://127.0.0.1:56000' processes=1 threads=1, memory=16.76 GiB> 
[2026-03-12 17:14:13,935]     INFO        main:  Time to create empty image xds: 0.0010771751403808594 seconds 
[2026-03-12 17:14:13,946]     INFO        main:  Time to write empty image to disk: 0.010747909545898438 seconds 


INFO:main:Time to write empty image to disk: 0.010747909545898438 seconds


[2026-03-12 17:14:13,946]     INFO        main:  Memory required for a single frequency channel: 0.005029141902923584 GiB 


INFO:main:Memory required for a single frequency channel: 0.005029141902923584 GiB


[2026-03-12 17:14:13,947]     INFO        main:  Thread info {'n_threads': 1, 'memory_per_thread': 16.763806343078613} 


...

[2026-03-12 17:14:14,975]    DEBUG    worker_0:  Memory usage at start of wrap_image_cube_single_field_node_task: 0.272449536 GB
[2026-03-12 17:14:14,980]    DEBUG    worker_0:  Processing set iterator created with partitions.
[2026-03-12 17:14:14,980]    DEBUG    worker_0:  Memory usage after completing node task, before releasing references: 0.277168128 GB
[2026-03-12 17:14:15,053]    DEBUG    worker_0:  Memory usage after releasing references: 0.301940736 GB
[2026-03-12 17:14:15,056]    DEBUG    worker_0:  Memory usage at start of wrap_image_cube_single_field_node_task: 0.302727168 GB
[2026-03-12 17:14:15,057]    DEBUG    worker_0:  Processing set iterator created with partitions.
[2026-03-12 17:14:15,057]    DEBUG    worker_0:  Memory usage after completing node task, before releasing references: 0.301776896 GB
[2026-03-12 17:14:15,106]    DEBUG    worker_0:  Memory usage after releasing references: 0.301776896 GB
[2026-03-12 17:14:15,111]    DEBUG    worker_0:  Memory usage at sta

INFO:main:Time to compute dask graph: 1.091548204421997 seconds


[2026-03-12 17:14:15,185]     INFO        main:  Time to consolidate metadata: 0.0022008419036865234 seconds 


INFO:main:Time to consolidate metadata: 0.0022008419036865234 seconds


<xarray.Dataset> Size: 25MB
Dimensions:                (time: 1, frequency: 5, polarization: 2, l: 250,
                            m: 250, beam_params_label: 3)
Coordinates:
  * time                   (time) float64 8B 0.0
  * frequency              (frequency) float64 40B 3.727e+11 ... 3.727e+11
    velocity               (frequency) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
  * polarization           (polarization) <U1 8B 'I' 'Q'
  * l                      (l) float64 2kB 6.06e-05 6.012e-05 ... -6.012e-05
  * m                      (m) float64 2kB -6.06e-05 -6.012e-05 ... 6.012e-05
  * beam_params_label      (beam_params_label) <U5 60B 'major' 'minor' 'pa'
Data variables:
    POINT_SPREAD_FUNCTION  (time, frequency, polarization, l, m) float64 5MB dask.array<chunksize=(1, 2, 2, 250, 250), meta=np.ndarray>
    PRIMARY_BEAM           (time, frequency, polarization, l, m) float64 5MB dask.array<chunksize=(1, 2, 2, 250, 250), meta=np.ndarray>
    SKY_DECONVOLVED        (time, frequency, polarization, l, m) float64 5MB dask.array<chunksize=(1, 2, 2, 250, 250), meta=np.ndarray>
    SKY_MODEL              (time, frequency, polarization, l, m) float64 5MB dask.array<chunksize=(1, 2, 2, 250, 250), meta=np.ndarray>
    SKY_RESIDUAL           (time, frequency, polarization, l, m) float64 5MB dask.array<chunksize=(1, 2, 2, 250, 250), meta=np.ndarray>
Attributes:
    coordinate_system_info:  {'native_pole_direction': {'attrs': {'frame': 'N...
    data_groups:             {'base': {}}
    type:                    image

In [ ]:
import toolviper.utils.logger as logger

In [ ]:
import xarray as xr
img_xds = xr.open_zarr("twhya_selfcal_5chans_lsrk_compare_weights.img.zarr")
img_xds

import matplotlib.pyplot as plt
plt.imshow(img_xds.PRIMARY_BEAM.isel(frequency=0, polarization=1, time=0).values)
plt.colorbar()

import matplotlib.pyplot as plt
plt.imshow(img_xds.POINT_SPREAD_FUNCTION.isel(frequency=0, polarization=1, time=0).values)
plt.colorbar()

# import matplotlib.pyplot as plt
# plt.imshow(img_xds.SKY.isel(frequency=0, polarization=1, time=0).values)
# plt.colorbar()